# SC-8-DeFi-Primitives - Primitives DeFi

[<< Token Standards](SC-7-Token-Standards.ipynb) | [DAO Governance >>](SC-9-DAO-Governance.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre les **AMM** (Automated Market Makers)
2. Implementer la formule **x*y=k** (Constant Product)
3. Creer des **liquidity pools** simples
4. Comprendre le **price impact** et le **slippage**

### Prerequis

- [SC-1](../01-Solidity-Foundation/SC-3-Solidity-Basics.ipynb) a [SC-5](SC-7-Token-Standards.ipynb) completes
- Notions de finance decentralisee (AMM, liquidite)
- Maitrise des interfaces et appels externes Solidity

### Duree estimee : 55 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


In [ ]:
# Connection a anvil (blockchain locale Foundry)
# Prerequis: anvil en cours d'execution dans un terminal
from web3 import Web3
import solcx

SOLC_VERSION = "0.8.28"
ANVIL_URL = "http://127.0.0.1:8545"

# Connexion
w3 = Web3(Web3.HTTPProvider(ANVIL_URL))
assert w3.is_connected(), f"Impossible de se connecter a {ANVIL_URL}. Lancez 'anvil' dans un terminal."

# Installer solc si necessaire
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

deployer = w3.eth.accounts[0]
print(f"Connecte a anvil (chain {w3.eth.chain_id}), deployer: {deployer[:10]}...")


def compile_and_deploy(w3, source_code, deployer, *constructor_args):
    """Compiler et deployer un contrat Solidity."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    contract_id, contract_interface = compiled.popitem()
    Contract = w3.eth.contract(
        abi=contract_interface["abi"], bytecode=contract_interface["bin"]
    )
    tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    instance = w3.eth.contract(
        address=receipt.contractAddress, abi=contract_interface["abi"]
    )
    print(f"Deploye: {contract_id.split(':')[-1]} a {receipt.contractAddress}")
    return instance, receipt


---

## 1. Automated Market Maker (AMM)

Les AMM remplacent les carnets d'ordres par des formules mathematiques.

### Formule Constant Product : $x \cdot y = k$

Ou :
- $x$ = reserve de token A
- $y$ = reserve de token B  
- $k$ = constante (ne change pas lors d'un swap)

In [1]:
# Simulation AMM en Python
def simulate_swap(reserve_a, reserve_b, amount_in, fee_percent=0.3):
    """
    Simule un swap selon la formule x*y=k
    
    Args:
        reserve_a: Reserve de token A
        reserve_b: Reserve de token B
        amount_in: Montant de A a echanger
        fee_percent: Fee en pourcentage
    
    Returns:
        amount_out: Montant de B recu
        new_reserve_a: Nouvelle reserve A
        new_reserve_b: Nouvelle reserve B
        price_impact: Impact sur le prix en %
    """
    # Calcul du fee
    amount_in_with_fee = amount_in * (1 - fee_percent / 100)
    
    # Constante k
    k = reserve_a * reserve_b
    
    # Nouvelles reserves
    new_reserve_a = reserve_a + amount_in_with_fee
    new_reserve_b = k / new_reserve_a
    
    # Montant recu
    amount_out = reserve_b - new_reserve_b
    
    # Prix avant/apres
    price_before = reserve_b / reserve_a
    price_after = new_reserve_b / new_reserve_a
    price_impact = ((price_before - price_after) / price_before) * 100
    
    return amount_out, new_reserve_a, new_reserve_b, price_impact

# Exemple
reserve_a = 1000  # ETH
reserve_b = 2000000  # USDC
amount_in = 10  # ETH a echanger

amount_out, new_a, new_b, impact = simulate_swap(reserve_a, reserve_b, amount_in)

print(f"Reserves initiales: {reserve_a} ETH / {reserve_b} USDC")
print(f"Swap: {amount_in} ETH -> {amount_out:.2f} USDC")
print(f"Nouvelles reserves: {new_a:.2f} ETH / {new_b:.2f} USDC")
print(f"Prix: {reserve_b/reserve_a:.2f} USDC/ETH -> {new_b/new_a:.2f} USDC/ETH")
print(f"Price impact: {impact:.2f}%")

Reserves initiales: 1000 ETH / 2000000 USDC
Swap: 10 ETH -> 19743.16 USDC
Nouvelles reserves: 1009.97 ETH / 1980256.84 USDC
Prix: 2000.00 USDC/ETH -> 1960.71 USDC/ETH
Price impact: 1.96%


In [2]:
# AMM Solidity implementation
AMM_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

interface IERC20 {
    function transferFrom(address from, address to, uint256 amount) external returns (bool);
    function transfer(address to, uint256 amount) external returns (bool);
    function balanceOf(address account) external view returns (uint256);
}

contract SimpleAMM {
    IERC20 public immutable token0;  // Ex: WETH
    IERC20 public immutable token1;  // Ex: USDC

    uint256 public reserve0;
    uint256 public reserve1;
    uint256 public totalLiquidity;

    mapping(address => uint256) public liquidity;

    uint256 private constant FEE_NUMERATOR = 3;
    uint256 private constant FEE_DENOMINATOR = 1000;  // 0.3%

    event Swap(
        address indexed sender,
        uint256 amount0In,
        uint256 amount1In,
        uint256 amount0Out,
        uint256 amount1Out
    );
    event LiquidityAdded(
        address indexed provider,
        uint256 amount0,
        uint256 amount1,
        uint256 liquidityMinted
    );
    event LiquidityRemoved(
        address indexed provider,
        uint256 amount0,
        uint256 amount1,
        uint256 liquidityBurned
    );

    constructor(address _token0, address _token1) {
        token0 = IERC20(_token0);
        token1 = IERC20(_token1);
    }

    // Ajouter de la liquidite
    function addLiquidity(
        uint256 amount0Desired,
        uint256 amount1Desired,
        uint256 amount0Min,
        uint256 amount1Min
    ) external returns (uint256 liquidityMinted) {
        // Transfer les tokens
        token0.transferFrom(msg.sender, address(this), amount0Desired);
        token1.transferFrom(msg.sender, address(this), amount1Desired);

        // Calcul de la liquidite a mint
        if (totalLiquidity == 0) {
            // Premier fournisseur: geometric mean
            liquidityMinted = sqrt(amount0Desired * amount1Desired);
        } else {
            // Proportionnel aux reserves existantes
            uint256 liquidity0 = (amount0Desired * totalLiquidity) / reserve0;
            uint256 liquidity1 = (amount1Desired * totalLiquidity) / reserve1;
            liquidityMinted = liquidity0 < liquidity1 ? liquidity0 : liquidity1;
        }

        require(liquidityMinted > 0, "Insufficient liquidity minted");
        require(amount0Desired >= amount0Min, "Slippage: amount0");
        require(amount1Desired >= amount1Min, "Slippage: amount1");

        liquidity[msg.sender] += liquidityMinted;
        totalLiquidity += liquidityMinted;
        reserve0 += amount0Desired;
        reserve1 += amount1Desired;

        emit LiquidityAdded(msg.sender, amount0Desired, amount1Desired, liquidityMinted);
    }

    // Retirer de la liquidite
    function removeLiquidity(uint256 liquidityToRemove) 
        external 
        returns (uint256 amount0, uint256 amount1) 
    {
        require(liquidity[msg.sender] >= liquidityToRemove, "Insufficient liquidity");

        amount0 = (liquidityToRemove * reserve0) / totalLiquidity;
        amount1 = (liquidityToRemove * reserve1) / totalLiquidity;

        liquidity[msg.sender] -= liquidityToRemove;
        totalLiquidity -= liquidityToRemove;
        reserve0 -= amount0;
        reserve1 -= amount1;

        token0.transfer(msg.sender, amount0);
        token1.transfer(msg.sender, amount1);

        emit LiquidityRemoved(msg.sender, amount0, amount1, liquidityToRemove);
    }

    // Swap token0 -> token1
    function swap0For1(uint256 amount0In, uint256 amount1OutMin) 
        external 
        returns (uint256 amount1Out) 
    {
        token0.transferFrom(msg.sender, address(this), amount0In);

        // Fee deduite
        uint256 amount0InWithFee = (amount0In * (FEE_DENOMINATOR - FEE_NUMERATOR)) / FEE_DENOMINATOR;

        // Constant product formula: (reserve0 + amountIn) * (reserve1 - amountOut) = k
        // amountOut = reserve1 - (reserve0 * reserve1) / (reserve0 + amountInWithFee)
        amount1Out = (amount0InWithFee * reserve1) / (reserve0 + amount0InWithFee);

        require(amount1Out >= amount1OutMin, "Slippage: amount1Out too low");
        require(amount1Out <= reserve1, "Insufficient reserve1");

        reserve0 += amount0In;
        reserve1 -= amount1Out;

        token1.transfer(msg.sender, amount1Out);

        emit Swap(msg.sender, amount0In, 0, 0, amount1Out);
    }

    // Swap token1 -> token0
    function swap1For0(uint256 amount1In, uint256 amount0OutMin) 
        external 
        returns (uint256 amount0Out) 
    {
        token1.transferFrom(msg.sender, address(this), amount1In);

        uint256 amount1InWithFee = (amount1In * (FEE_DENOMINATOR - FEE_NUMERATOR)) / FEE_DENOMINATOR;
        amount0Out = (amount1InWithFee * reserve0) / (reserve1 + amount1InWithFee);

        require(amount0Out >= amount0OutMin, "Slippage: amount0Out too low");
        require(amount0Out <= reserve0, "Insufficient reserve0");

        reserve1 += amount1In;
        reserve0 -= amount0Out;

        token0.transfer(msg.sender, amount0Out);

        emit Swap(msg.sender, 0, amount1In, amount0Out, 0);
    }

    // Prix actuel
    function getPrice0() external view returns (uint256) {
        require(reserve0 > 0, "No liquidity");
        return (reserve1 * 1e18) / reserve0;
    }

    // Square root function (Babylonian method)
    function sqrt(uint256 y) internal pure returns (uint256 z) {
        if (y > 3) {
            z = y;
            uint256 x = y / 2 + 1;
            while (x < z) {
                z = x;
                x = (y / x + x) / 2;
            }
        } else if (y != 0) {
            z = 1;
        }
    }
}
'''


# Deployer (ajuster les arguments du constructeur si necessaire)
# simpleamm, receipt = compile_and_deploy(w3, AMM_CONTRACT, deployer)
# print(f"Deploye a: {simpleamm.address}")

Implementation AMM complete:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

interface IERC20 {
    function transferFrom(address from, address to, uint256 amount) external returns (bool);
    function transfer(address to, uint256 amount) external returns (bool);
    function balanceOf(address account) external view returns (uint256);
}

contract SimpleAMM {
    IERC20 public immutable token0;  // Ex: WETH
    IERC20 public immutable token1;  // Ex: USDC

    uint256 public reserve0;
    uint256 public reserve1;
    uint256 public totalLiquidity;

    mapping(address => uint256) public liquidity;

    uint256 private constant FEE_NUMERATOR = 3;
    uint256 private constant FEE_DENOMINATOR = 1000;  // 0.3%

    event Swap(
        address indexed sender,
        uint256 amount0In,
        uint256 amount1In,
        uint256 amount0Out,
        uint256 amount1Out
    );
    event LiquidityAdded(
        address indexed provider,
        uint256 amount0,
        uint256 amount1,

---

## 2. Price Impact et Slippage

- **Price Impact** : Changement de prix cause par votre trade
- **Slippage** : Difference entre le prix attendu et obtenu

In [3]:
# Visualisation du price impact
def calculate_price_impact_curve(reserve_a, reserve_b, max_swap_percent=50):
    """Calcule le price impact pour differentes tailles de swap"""
    import math
    
    k = reserve_a * reserve_b
    initial_price = reserve_b / reserve_a
    
    results = []
    for swap_percent in range(1, max_swap_percent + 1):
        amount_in = reserve_a * swap_percent / 100
        amount_out, new_a, new_b, impact = simulate_swap(reserve_a, reserve_b, amount_in)
        new_price = new_b / new_a
        
        results.append({
            'swap_percent': swap_percent,
            'amount_in': amount_in,
            'amount_out': amount_out,
            'price_impact': impact,
            'execution_price': amount_out / amount_in,
            'spot_price': new_price
        })
    
    return results

# Affichage
results = calculate_price_impact_curve(1000, 2000000)
print("Swap% | Amount In | Amount Out | Price Impact | Exec Price")
print("-" * 65)
for r in results[::10]:  # Every 10%
    print(f"{r['swap_percent']:5}% | {r['amount_in']:9.1f} | {r['amount_out']:10.2f} | {r['price_impact']:11.2f}% | {r['execution_price']:.2f}")

Swap% | Amount In | Amount Out | Price Impact | Exec Price
-----------------------------------------------------------------
    1% |      10.0 |   19743.16 |        1.96% | 1974.32
   11% |     110.0 |  197662.37 |       18.79% | 1796.93
   21% |     210.0 |  346246.39 |       31.63% | 1648.79
   31% |     310.0 |  472197.82 |       41.65% | 1523.22
   41% |     410.0 |  580321.84 |       49.61% | 1415.42


---

## 3. Lending Simplifie

Principes du lending DeFi :
- Deposer des collateraux
- Emprunter contre collateral
- Taux d'interet dynamiques

In [4]:
# Lending Pool simplifie
LENDING_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract SimpleLending {
    IERC20 public immutable asset;
    
    // Utilisation: montant emprunte / montant depose
    uint256 public constant MAX_LTV = 75;  // 75%
    uint256 public constant LIQUIDATION_THRESHOLD = 85;  // 85%
    uint256 public constant LIQUIDATION_BONUS = 10;  // 10%
    
    // Taux d'interet (simplifie)
    uint256 public interestRate = 5;  // 5% APY
    
    struct UserAccount {
        uint256 collateral;      // Montant depose
        uint256 borrowed;        // Montant emprunte
        uint256 lastUpdate;      // Derniere mise a jour
    }
    
    mapping(address => UserAccount) public accounts;
    
    event Deposit(address indexed user, uint256 amount);
    event Borrow(address indexed user, uint256 amount);
    event Repay(address indexed user, uint256 amount);
    event Liquidation(
        address indexed liquidator,
        address indexed borrower,
        uint256 collateralSeized,
        uint256 debtRepaid
    );

    constructor(address _asset) {
        asset = IERC20(_asset);
    }

    // Deposer du collateral
    function deposit(uint256 amount) external {
        asset.transferFrom(msg.sender, address(this), amount);
        _accrueInterest(msg.sender);
        accounts[msg.sender].collateral += amount;
        emit Deposit(msg.sender, amount);
    }

    // Emprunter
    function borrow(uint256 amount) external {
        _accrueInterest(msg.sender);
        UserAccount storage user = accounts[msg.sender];
        
        uint256 maxBorrow = (user.collateral * MAX_LTV) / 100;
        require(user.borrowed + amount <= maxBorrow, "Insufficient collateral");
        
        user.borrowed += amount;
        asset.transfer(msg.sender, amount);
        emit Borrow(msg.sender, amount);
    }

    // Rembourser
    function repay(uint256 amount) external {
        _accrueInterest(msg.sender);
        UserAccount storage user = accounts[msg.sender];
        
        require(amount <= user.borrowed, "Amount exceeds debt");
        
        asset.transferFrom(msg.sender, address(this), amount);
        user.borrowed -= amount;
        emit Repay(msg.sender, amount);
    }

    // Retirer du collateral
    function withdraw(uint256 amount) external {
        _accrueInterest(msg.sender);
        UserAccount storage user = accounts[msg.sender];
        
        require(amount <= user.collateral, "Insufficient collateral");
        
        // Verifier LTV apres retrait
        uint256 remainingCollateral = user.collateral - amount;
        uint256 maxBorrow = (remainingCollateral * MAX_LTV) / 100;
        require(user.borrowed <= maxBorrow, "Would breach LTV");
        
        user.collateral -= amount;
        asset.transfer(msg.sender, amount);
    }

    // Liquidation
    function liquidate(address borrower) external {
        _accrueInterest(borrower);
        UserAccount storage user = accounts[borrower];
        
        // Verifier si sous-collateralise
        uint256 collateralValue = user.collateral;
        uint256 maxAllowedDebt = (collateralValue * LIQUIDATION_THRESHOLD) / 100;
        
        require(user.borrowed > maxAllowedDebt, "Position healthy");
        
        // Calculer montant a liquider (50% de la dette max)
        uint256 debtToRepay = user.borrowed / 2;
        uint256 collateralToSeize = (debtToRepay * (100 + LIQUIDATION_BONUS)) / 100;
        
        // Verifier qu'on ne prend pas plus que disponible
        if (collateralToSeize > user.collateral) {
            collateralToSeize = user.collateral;
        }
        
        // Transferts
        asset.transferFrom(msg.sender, address(this), debtToRepay);
        user.borrowed -= debtToRepay;
        user.collateral -= collateralToSeize;
        asset.transfer(msg.sender, collateralToSeize);
        
        emit Liquidation(msg.sender, borrower, collateralToSeize, debtToRepay);
    }

    // Calcul interets
    function _accrueInterest(address user) internal {
        UserAccount storage account = accounts[user];
        if (account.borrowed == 0 || account.lastUpdate == 0) {
            account.lastUpdate = block.timestamp;
            return;
        }
        
        uint256 timeElapsed = block.timestamp - account.lastUpdate;
        uint256 interest = (account.borrowed * interestRate * timeElapsed) / (100 * 365 days);
        
        account.borrowed += interest;
        account.lastUpdate = block.timestamp;
    }

    // Getters
    function getHealthFactor(address user) external view returns (uint256) {
        UserAccount memory account = accounts[user];
        if (account.borrowed == 0) return type(uint256).max;
        return (account.collateral * LIQUIDATION_THRESHOLD * 1e18) / (account.borrowed * 100);
    }
}
'''


# Deployer (ajuster les arguments du constructeur si necessaire)
# simplelending, receipt = compile_and_deploy(w3, LENDING_CONTRACT, deployer)
# print(f"Deploye a: {simplelending.address}")

Contrat de Lending simplifie:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract SimpleLending {
    IERC20 public immutable asset;

    // Utilisation: montant emprunte / montant depose
    uint256 public constant MAX_LTV = 75;  // 75%
    uint256 public constant LIQUIDATION_THRESHOLD = 85;  // 85%
    uint256 public constant LIQUIDATION_BONUS = 10;  // 10%

    // Taux d'interet (simplifie)
    uint256 public interestRate = 5;  // 5% APY

    struct UserAccount {
        uint256 collateral;      // Montant depose
        uint256 borrowed;        // Montant emprunte
        uint256 lastUpdate;      // Derniere mise a jour
    }

    mapping(address => UserAccount) public accounts;

    event Deposit(address indexed user, uint256 amount);
    event Borrow(address indexed user, uint256 amount);
    event Repay(address indexed user, uint256 amount);
    event Liquidation(
        address indexed liquidator,
        address indexed borrower,
        uint256 collateralSeiz

---

## 4. Exercices

### Exercice 1 : Calculer le montant optimal de swap

Ecrivez une fonction qui calcule combien de token B on recoit pour un swap de X token A.

In [ ]:
# Exercice 1 : Calculer le montant optimal de swap
def get_amount_out(amount_in, reserve_in, reserve_out, fee=0.003):
    """
    Calcule le montant de sortie selon x*y=k
    
    Formule: amountOut = (amountIn * (1-fee) * reserveOut) / (reserveIn + amountIn * (1-fee))
    
    Args:
        amount_in: Montant de token A a echanger
        reserve_in: Reserve de token A dans le pool
        reserve_out: Reserve de token B dans le pool
        fee: Fee en fraction (0.003 = 0.3%)
    
    Returns:
        float: Montant de token B recu
    """
    # TODO: Calculer amount_in apres deduction du fee
    # TODO: Calculer le numerateur (amount_in_with_fee * reserve_out)
    # TODO: Calculer le denominateur (reserve_in + amount_in_with_fee)
    # TODO: Retourner le resultat
    pass

# Test (decommenter apres implementation)
# print(f"Swap 10 ETH: {get_amount_out(10, 1000, 2000000):.2f} USDC")
# print(f"Swap 100 ETH: {get_amount_out(100, 1000, 2000000):.2f} USDC")
# print(f"Swap 500 ETH: {get_amount_out(500, 1000, 2000000):.2f} USDC")

### Exercice 2 : Calcul du Health Factor

Implementez le calcul du health factor pour un protocol de lending.

In [ ]:
# Exercice 2 : Calcul du Health Factor
def calculate_health_factor(collateral, borrowed, liquidation_threshold=0.85):
    """
    Health Factor = (Collateral * LiquidationThreshold) / Borrowed
    
    - HF > 1: Position safe
    - HF = 1: Limite de liquidation
    - HF < 1: Position liquidable
    
    Args:
        collateral: Montant du collateral depose
        borrowed: Montant emprunte
        liquidation_threshold: Seuil de liquidation (0.85 = 85%)
    
    Returns:
        float: Health factor
    """
    # TODO: Gerer le cas ou borrowed == 0 (retourner float('inf'))
    # TODO: Calculer et retourner (collateral * liquidation_threshold) / borrowed
    pass

# Test (decommenter apres implementation)
# test_cases = [
#     (1000, 500),   # Safe
#     (1000, 800),   # Close to limit
#     (1000, 850),   # At limit
#     (1000, 900),   # Liquidable
# ]
# 
# print("Collateral | Borrowed | Health Factor | Status")
# print("-" * 55)
# for c, b in test_cases:
#     hf = calculate_health_factor(c, b)
#     status = "Safe" if hf > 1 else ("At limit" if hf == 1 else "Liquidable")
#     print(f"{c:10} | {b:8} | {hf:13.4f} | {status}")

---

## 5. Resume

| Concept | Formule | Description |
|---------|---------|-------------|
| Constant Product | $x \cdot y = k$ | Base des AMM |
| Swap Output | $\frac{\Delta x \cdot y}{x + \Delta x}$ | Montant recu |
| Price Impact | $\frac{P_{avant} - P_{apres}}{P_{avant}}$ | Impact du trade |
| Health Factor | $\frac{Collateral \cdot LT}{Dette}$ | Securite lending |

### Points cles
1. Les AMM remplacent les order books par des formules mathematiques
2. Le price impact augmente avec la taille du swap
3. Toujours definir un slippage tolerance
4. Le health factor < 1 declenche la liquidation

---

**Notebook suivant** : [SC-9-DAO-Governance](SC-9-DAO-Governance.ipynb)

---

[<< Token Standards](SC-7-Token-Standards.ipynb) | [DAO Governance >>](SC-9-DAO-Governance.ipynb)